# E5 · Evaluación del modelo v0 (OCR de pantallas)
**Capstone Project I · Equipo 2** · Christian Cordova, Meliza Sosa, Yesseliz Choque · Asesor: Boris Alzamora

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JesselizCB/asistente-salud/blob/main/notebooks/E5_evaluacion.ipynb)

Reproduce todos los números de la E5 a partir de los CSV de predicciones, sin necesitar las fotos:
1. Tabla baseline contra modelo v0 (exactitud por foto y % de fotos con valor errado, con IC 95 %).
2. Diferencia v0 − modelo simple con bootstrap pareado (5 000 remuestreos, semilla 2026).
3. Error por segmento de glucosa.
4. Segunda muestra ACC-1.

**Datos.** Fotos públicas de Kaggle (DataCluster Labs, 2021): 120 de glucómetro y 120 de tensiómetro, con sus etiquetas. No son fotos tomadas por el equipo. Las fotos no están en el repositorio (≈ 640 MB); para volver a correr el OCR ver el README.

In [1]:
# En Colab: clona el repositorio e instala dependencias. En local no hace nada.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone -q https://github.com/JesselizCB/asistente-salud.git
    %cd asistente-salud
    !pip -q install statsmodels
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import pandas as pd
import metricas as m
pd.set_option("display.width", 200)
print("Semilla:", m.SEMILLA, "· remuestreos:", m.N_BOOT)

Semilla: 2026 · remuestreos: 5000


## 1 · Baseline contra modelo v0

In [2]:
g, t = m.cargar("data")
tabla = m.tabla_baseline(g, t)
tabla

,aparato,modelo,n,aciertos,exactitud,ic95,errados,pct_errado,ic95_errado
0,glucometro,simple,120,66.0,55.0,"(46.1, 63.6)",18.0,15.0,"(9.7, 22.5)"
1,glucometro,v0,120,96.0,80.0,"(72.0, 86.2)",16.0,13.3,"(8.4, 20.6)"
2,glucometro,v0 - simple (pp),120,NaN,25.0,"(15.8, 34.2)",NaN,NaN,NaN
3,tensiometro,simple,120,3.0,2.5,"(0.9, 7.1)",71.0,59.2,"(50.2, 67.5)"
4,tensiometro,v0,120,8.0,6.7,"(3.4, 12.6)",33.0,27.5,"(20.3, 36.1)"
5,tensiometro,v0 - simple (pp),120,NaN,4.2,"(-1.7, 10.0)",NaN,NaN,NaN


**Lectura.** En el glucómetro el v0 pasa de 55,0 % a 80,0 % y la diferencia (+25,0 pp) tiene un IC que no incluye el 0. En el tensiómetro la diferencia (+4,2 pp) sí incluye el 0: no hay mejora clara.

La lectura humana (95,0 % [88,8–97,8]) se midió en otro conjunto, la muestra ACC-1 del E4; la latencia se midió con `src/latencia_v0.py` (mediana 4,0 s glucómetro y 4,6 s tensiómetro en una Mac M1).

## 2 · Dónde falla el v0: segmentos de glucosa

In [3]:
seg = m.segmentos_glucosa(g)
seg

,aciertos,n,exactitud,ic95
rango,,,,
< 100,13,17,76.5,"(52.7, 90.4)"
100-180,58,68,85.3,"(75.0, 91.8)"
> 180,25,35,71.4,"(54.9, 83.7)"


In [4]:
# Tamaño de los errores del v0 en el glucómetro
err = g[g.mal_v0]
print("Fotos con valor errado:", len(err), "· errores de 50 mg/dL o más:", int(((err.g_v0 - err.label_glucose).abs() >= 50).sum()))
err[["filename", "g_v0", "label_glucose"]]

Fotos con valor errado: 16 · errores de 50 mg/dL o más: 14


,filename,g_v0,label_glucose
8,20210808_09_45_48_000_SrR1euGc9XWObcMVhApCggGM...,61.0,194
10,20210808_20_49_47_136_000_qg2NDAhSeCX7kAEWB0Jv...,20.0,201
16,20210810_15_27_37_763_000_eFS2rWFgvKf87TdgWVI0...,305.0,161
17,20210810_19_41_44_993_000_GA0fDyUPwUeWo8q3qKTr...,66.0,99
18,20210810_21_45_00_150_000_5XIWzz9zhpWeOWO1ulYX...,621.0,129
38,20210816_06_39_54_099_000_TLA5Quo5ujdx8G4F6QH7...,11.0,110
42,20210817_12_35_03_859_000_Xyfrcq8vF5XqwmDi1zv8...,260.0,262
46,20210823_05_40_03_157_000_rUsQWUJGXwZr1da69AP3...,82.0,199
52,20210823_20_59_42_354_000_3FQ7xpkMmcdRS27U2bBC...,0.0,173
81,20210829_11_48_57_060_000_wmO8RFoaLcbLFpr0Ject...,19.0,191


## 3 · Segunda muestra ACC-1
100 fotos del mismo dataset (60 de glucómetro y 40 de tensiómetro) con la pantalla completa y enfocada, porque en el producto la guía por voz pide encuadrar antes de leer. Por ese criterio de selección sus resultados pueden ser optimistas; no es una prueba externa.

In [5]:
acc1 = pd.read_csv("data/etiquetas/acc1_muestra.csv")
for nombre, d in [("glucometro", g), ("tensiometro", t)]:
    s = d[d.filename.isin(acc1.filename)]
    k, n = int(s.ok_v0.sum()), len(s)
    print(f"{nombre}: v0 {k}/{n} = {100*k/n:.1f} % IC95 {m.wilson(k, n)}")

glucometro: v0 48/60 = 80.0 % IC95 (68.2, 88.2)
tensiometro: v0 4/40 = 10.0 % IC95 (4.0, 23.1)


## 4 · Verificación: los números coinciden con la E5

In [6]:
fila = lambda a, mo: tabla[(tabla.aparato == a) & (tabla.modelo == mo)].iloc[0]
assert fila("glucometro", "v0").aciertos == 96 and fila("glucometro", "v0").ic95 == (72.0, 86.2)
assert fila("glucometro", "simple").aciertos == 66
assert fila("glucometro", "v0 - simple (pp)").ic95 == (15.8, 34.2)
assert fila("tensiometro", "v0").aciertos == 8 and fila("tensiometro", "v0").errados == 33
assert fila("tensiometro", "v0 - simple (pp)").ic95 == (-1.7, 10.0)
print("OK: todos los números coinciden con la E5")

OK: todos los números coinciden con la E5


## Limitaciones
- Fotos de terceros, no tomadas con la tablet ni por adultos mayores; ninguna glucosa menor que 70 mg/dL.
- Las reglas del v0 se ajustaron mirando estas mismas fotos: el resultado puede ser optimista.
- La validación externa será con fotos tomadas por el equipo al 14 de noviembre de 2026.